## FUNCTIONAL PROGRAMMING - FINAL PROJECT 26-27
José Manuel Cosano (100523050) & Mateo Nonzioli (100523033)

1. Type definitions for all concepts in the statement

In [ ]:
-- Descriptors
type Description = String
data Type = Private | Doctors | Work deriving (Show, Eq)

-- Date definition
-- Dedicated data type for the month 
data Month = January | February | March | April
    | May | June | July | August
    | September | October | November | December deriving (Eq, Enum)

instance Show Month where
    show January   = "January"
    show February  = "February"
    show March     = "March"
    show April     = "April"
    show May       = "May"
    show June      = "June"
    show July      = "July"
    show August    = "August"
    show September = "September"
    show October   = "October"
    show November  = "November"
    show December  = "December"

data IsWorkDay = WorkingDay | Holiday deriving (Show, Eq)

-- Date follows the format (Day Month Year IsWorkDay) for valid dates with Day and Year integers and Month as above
data Date = ValidDate Int Month Int IsWorkDay | InvalidDate deriving Eq

-- Override Show instance for Date (accounting for invalid dates)
instance Show Date where
    show InvalidDate = "Invalid date"
    show (ValidDate d m y isworkday) =
        -- Here "show m" uses the Show Month instance defined above
        show d ++ " " ++ show m ++ " " ++ show y ++ " (" ++ show isworkday ++ ")" 

-- Time definition. We use the Maybe type to allow for optional arguments
type HoursMinutesTime = (Int, Int)
data Time = Time {start :: Maybe HoursMinutesTime, end :: Maybe HoursMinutesTime, duration :: Maybe Int} deriving Eq

-- Override Show instance for Time
-- For start and end, show already handles Maybe, since HoursMinutesTime is made of two Int (which have Show instances)
instance Show Time where
    show (Time start end duration) = 
        "Time { start = " ++ show start ++ ", end = " ++ show end ++ ", duration = " ++ show duration ++ " }"

-- List of participants
newtype Person = Person String deriving (Show, Eq)
type Participants = [Person]

-- Frequency of the event for REPEATED events ONLY (daily, weekly, working days)
data Repetition = Daily | Weekly | WorkDays deriving (Show, Eq)

-- Distinguish between Punctual and Repeated events
data Repetitions = Punctual | Repeated Repetition Date deriving (Show, Eq)

-- Complete constructor for events
data Event = Event
    { description :: Description
    , eventType :: Type
    , date :: Maybe Date -- Date is specified as optional
    , time :: Time
    , participants :: Participants
    , repetitions :: Repetitions
    } deriving Eq

2. Function to display events in a pretty way

In [ ]:
-- Override Show class for Event

4. completeTime function that completes the timing of an Event: if it lacks a duration, it calculates and
inserts it from the start and end times; if it has a start time and duration, it calculates the end time, and
vice versa. If it only specifies a duration, it leaves the Event unchanged as it is not yet scheduled.

In [ ]:
-- Auxiliary functions to convert time to/from minutes
toMinutes :: HoursMinutesTime -> Int
toMinutes (h, m) = 60 * h + m 

fromMinutes :: Int -> Maybe HoursMinutesTime
fromMinutes timeMin 
    | timeMin < 0 || timeMin >= 24*60 = Nothing
    | otherwise = Just (timeMin `div` 60, timeMin `mod` 60) 
    -- We use `div` (integer division) to display the hour as an Int, with the minutes displayed separately

-- Separate function to complete the time whenever needed
completeEvTime :: Time -> Time

-- Use pattern matching to dist
-- start and end times provided => convert to minutes and subtract
completeEvTime t@(Time (Just start) (Just end) Nothing)
    | computedDuration >= 0 = t {duration = Just computedDuration}
    | otherwise = t
    where
        computedDuration = toMinutes end - toMinutes start

-- start and duration provided
completeEvTime t@(Time (Just start) Nothing (Just duration))
    | duration >= 0 = t {end = fromMinutes (toMinutes start + duration)}
    | otherwise = t

-- end and duration provided
completeEvTime t@(Time Nothing (Just end) (Just duration))
    | duration >= 0 = t {start = fromMinutes (toMinutes end - duration)}
    | otherwise = t

-- in any other case, return the same time
completeEvTime t = t

completeTime :: Event -> Event
completeTime ev = ev {time = completeEvTime(time ev)}
